Hybrid Music Recommender



In [ ]:
!pip install -q implicit pandas numpy scikit-learn requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 137.0 MB/s eta 0:00:00


In [ ]:
from implicit.bpr import BayesianPersonalizedRanking

/usr/local/lib/python3.13/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pandas as pd
WORKDIR = "/content/drive/MyDrive/hybrid_music_rec"
os.makedirs(WORKDIR, exist_ok=True)
print("Working directory:", WORKDIR)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/hybrid_music_rec


In [ ]:
WORKDIR = "/content/drive/MyDrive/hybrid_music_rec"
SUBSAMPLE_DIR = os.path.join(WORKDIR, "processed")
CS_RESULTS = os.path.join(SUBSAMPLE_DIR, "cold_start_results.csv")

In [ ]:
import os

RAW_DIR = WORKDIR

print("Files found in RAW_DIR:")
for fname in os.listdir(RAW_DIR):
    fpath = os.path.join(RAW_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {fname}  ({size_mb:.1f} MB)")


Files found in RAW_DIR:
  id_genres_tf-idf.tsv.bz2  (2.8 MB)
  id_mfcc_stats.tsv.bz2  (89.4 MB)
  userid_trackid_count.tsv.bz2  (525.1 MB)
  README.md  (0.0 MB)


In [ ]:
import os

interactions_path = os.path.join(RAW_DIR, "userid_trackid_count.tsv.bz2")
mfcc_path = os.path.join(RAW_DIR, "id_mfcc_stats.tsv.bz2")
genres_path = os.path.join(RAW_DIR, "id_genres_tf-idf.tsv.bz2")

import bz2

for name, path in [("interactions", interactions_path), ("mfcc", mfcc_path), ("genres", genres_path)]:
    print(f"--- {name} ---")
    with bz2.open(path, "rt") as f:
        for _ in range(2):
            print(f.readline().strip()[:200])
    print()

--- interactions ---
user_id	track_id	count
92915	2fwmmHiKaZqDUlY0	1

--- mfcc ---
id	MFCC000	MFCC001	MFCC002	MFCC003	MFCC004	MFCC005	MFCC006	MFCC007	MFCC008	MFCC009	MFCC010	MFCC011	MFCC012	cov_0_0	cov_0_1	cov_0_2	cov_0_3	cov_0_4	cov_0_5	cov_0_6	cov_0_7	cov_0_8	cov_0_9	cov_0_10	cov_
9EPCgcAlXdgiFsx4	20.64096069335937	-5.828413009643555	-6.14085865020752	2.52666449546814	-1.4095522165298462	2.681575059890747	-2.98903489112854	6.298501014709473	3.5287089347839355	2.1831891536712646

--- genres ---
id	new weird america	flamenco	gospel	folk	breakbeat	jazz trumpet	funeral doom	underground hip hop	jazz metal	british folk	celtic metal	polish black metal	baroque	singer-songwriter	progressive post-har
0009fFIM1eYThaPg	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0



In [ ]:
import pandas as pd

interactions_df = pd.read_csv(interactions_path, sep="\t", compression="bz2")
mfcc_df = pd.read_csv(mfcc_path, sep="\t", compression="bz2")
genres_df = pd.read_csv(genres_path, sep="\t", compression="bz2")

#align ID column names so joins work later
mfcc_df = mfcc_df.rename(columns={"id": "track_id"})
genres_df = genres_df.rename(columns={"id": "track_id"})

print("Interactions:", interactions_df.shape)
print("MFCC:", mfcc_df.shape)
print("Genres:", genres_df.shape)

interactions_df.head()

Interactions: (50016042, 3)
MFCC: (109269, 105)
Genres: (109269, 686)


,user_id,track_id,count
0,92915,2fwmmHiKaZqDUlY0,1
1,92915,b4rHYoJ9wWDSZT3M,1
2,92915,LF0WX7weWu3LNbjC,2
3,92915,9AwpGifIAWQwGWZG,2
4,92915,xr1HkbplUycZcEZX,1


In [ ]:
n_users = interactions_df["user_id"].nunique()
n_items = interactions_df["track_id"].nunique()
n_interactions = len(interactions_df)

print(f"Users: {n_users:,}  (README says 119,140)")
print(f"Items: {n_items:,}  (README says 56,512)")
print(f"Interactions: {n_interactions:,}  (README says 50,016,042)")

Users: 119,140  (README says 119,140)
Items: 56,512  (README says 56,512)
Interactions: 50,016,042  (README says 50,016,042)


In [ ]:
N_USERS = 2000
MIN_INTERACTIONS_PER_USER = 5

USER_COL = "user_id"
ITEM_COL = "track_id"

#Keep only items that have MFCC audio features available
valid_items = set(mfcc_df[ITEM_COL])
filtered = interactions_df[interactions_df[ITEM_COL].isin(valid_items)]

#Keep users with enough interactions
user_counts = filtered[USER_COL].value_counts()
eligible_users = user_counts[user_counts >= MIN_INTERACTIONS_PER_USER].index

#Sample top-N most active eligible users
top_users = user_counts.loc[eligible_users].sort_values(ascending=False).head(N_USERS).index
subsample = filtered[filtered[USER_COL].isin(top_users)].reset_index(drop=True)

print(f"Subsample: {subsample[USER_COL].nunique():,} users, "
      f"{subsample[ITEM_COL].nunique():,} items, {len(subsample):,} interactions")

Subsample: 2,000 users, 56,377 items, 6,871,410 interactions


In [ ]:
SUBSAMPLE_DIR = os.path.join(WORKDIR, "processed")
os.makedirs(SUBSAMPLE_DIR, exist_ok=True)

subsample.to_csv(os.path.join(SUBSAMPLE_DIR, "interactions_subsample.csv"), index=False)

subsample_items = subsample[ITEM_COL].unique()
mfcc_subsample = mfcc_df[mfcc_df[ITEM_COL].isin(subsample_items)]
genres_subsample = genres_df[genres_df[ITEM_COL].isin(subsample_items)]

mfcc_subsample.to_csv(os.path.join(SUBSAMPLE_DIR, "mfcc_subsample.csv"), index=False)
genres_subsample.to_csv(os.path.join(SUBSAMPLE_DIR, "genres_subsample.csv"), index=False)

print("Saved:")
print(" -", os.path.join(SUBSAMPLE_DIR, "interactions_subsample.csv"), f"({len(subsample):,} rows)")
print(" -", os.path.join(SUBSAMPLE_DIR, "mfcc_subsample.csv"), f"({len(mfcc_subsample):,} rows)")
print(" -", os.path.join(SUBSAMPLE_DIR, "genres_subsample.csv"), f"({len(genres_subsample):,} rows)")

Saved:
 - /content/drive/MyDrive/hybrid_music_rec/processed/interactions_subsample.csv (6,871,410 rows)
 - /content/drive/MyDrive/hybrid_music_rec/processed/mfcc_subsample.csv (56,377 rows)
 - /content/drive/MyDrive/hybrid_music_rec/processed/genres_subsample.csv (56,377 rows)


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

subsample = pd.read_csv(os.path.join(SUBSAMPLE_DIR, "interactions_subsample.csv"))

#implicit needs integer indices, not string IDs — build the mapping
user_cat = subsample["user_id"].astype("category")
item_cat = subsample["track_id"].astype("category")

subsample["user_idx"] = user_cat.cat.codes
subsample["item_idx"] = item_cat.cat.codes

n_users = subsample["user_idx"].nunique()
n_items = subsample["item_idx"].nunique()
print(f"Encoded: {n_users:,} users, {n_items:,} items")

# Keep the mapping so you can convert indices back to real IDs later
user_id_map = dict(enumerate(user_cat.cat.categories))
item_id_map = dict(enumerate(item_cat.cat.categories))

Encoded: 2,000 users, 56,377 items


In [ ]:
MODEL_DIR = os.path.join(WORKDIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
from implicit.evaluation import train_test_split

full_matrix = csr_matrix(
    (subsample["count"].astype(np.float32), (subsample["user_idx"], subsample["item_idx"])),
    shape=(n_users, n_items)
)

#80/20 split of interactions, stratified per user under the hood
train_matrix, test_matrix = train_test_split(full_matrix, train_percentage=0.8, random_state=42)

print("Train interactions:", train_matrix.nnz)
print("Test interactions:", test_matrix.nnz)

Train interactions: 5497252
Test interactions: 1374158


In [ ]:
from implicit.bpr import BayesianPersonalizedRanking

model = BayesianPersonalizedRanking(
    factors=64,          # embedding dimension — reasonable starting point
    learning_rate=0.01,
    regularization=0.01,
    iterations=100,
    random_state=42
)

model.fit(train_matrix)
print("Training complete.")

  0%|          | 0/100 [00:00<?, ?it/s]

Training complete.


In [ ]:
from implicit.evaluation import precision_at_k, ndcg_at_k

K = 10

precision = precision_at_k(model, train_matrix, test_matrix, K=K)
ndcg = ndcg_at_k(model, train_matrix, test_matrix, K=K)

print(f"Precision@{K}: {precision:.4f}")
print(f"NDCG@{K}: {ndcg:.4f}")

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

Precision@10: 0.5391
NDCG@10: 0.5580


In [ ]:
def recall_at_k(model, train_matrix, test_matrix, K=10):
    recalls = []
    test_matrix = test_matrix.tocsr()
    for user_idx in range(test_matrix.shape[0]):
        test_items = set(test_matrix.indices[test_matrix.indptr[user_idx]:test_matrix.indptr[user_idx+1]])
        if not test_items:
            continue
        recommended, _ = model.recommend(
            user_idx, train_matrix[user_idx], N=K, filter_already_liked_items=True
        )
        hits = len(set(recommended) & test_items)
        recalls.append(hits / len(test_items))
    return np.mean(recalls) if recalls else 0.0

recall = recall_at_k(model, train_matrix, test_matrix, K=K)
print(f"Recall@{K}: {recall:.4f}")

Recall@10: 0.0082


In [ ]:
results = {
    "model": "Pure CF (implicit BPR)",
    "K": K,
    "precision_at_k": precision,
    "ndcg_at_k": ndcg,
    "recall_at_k": recall,
    "n_users": n_users,
    "n_items": n_items,
    "train_interactions": train_matrix.nnz,
    "test_interactions": test_matrix.nnz,
}

results_df = pd.DataFrame([results])
results_df.to_csv(os.path.join(SUBSAMPLE_DIR, "baseline_pure_cf_results.csv"), index=False)
print(results_df)

                    model   K  precision_at_k  ndcg_at_k  recall_at_k  \
0  Pure CF (implicit BPR)  10          0.5391    0.55796     0.008171   

   n_users  n_items  train_interactions  test_interactions  
0     2000    56377             5497252            1374158  


In [ ]:
#Load mfcc
import numpy as np
import pandas as pd

mfcc_subsample = pd.read_csv(os.path.join(SUBSAMPLE_DIR, "mfcc_subsample.csv"))
feature_cols = [c for c in mfcc_subsample.columns if c != "track_id"]
mfcc_subsample = mfcc_subsample.set_index("track_id")

# Align audio features to the same item_idx ordering used by the CF baseline
item_features = np.zeros((n_items, len(feature_cols)), dtype=np.float32)
missing = 0
for idx, track_id in item_id_map.items():
    if track_id in mfcc_subsample.index:
        item_features[idx] = mfcc_subsample.loc[track_id, feature_cols].values
    else:
        missing += 1

print(f"Item feature matrix: {item_features.shape}, {missing} items missing features (expect 0)")

Item feature matrix: (56377, 104), 0 items missing features (expect 0)


In [ ]:
#normalise features and buld NN index
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

scaler = StandardScaler()
item_features_scaled = scaler.fit_transform(item_features)

nn_model = NearestNeighbors(n_neighbors=50, metric="cosine")
nn_model.fit(item_features_scaled)
print("k-NN index built.")

k-NN index built.


In [ ]:
def recommend_content_based(user_idx, train_matrix, item_features_scaled, nn_model, K=10):
    train_items = train_matrix.indices[train_matrix.indptr[user_idx]:train_matrix.indptr[user_idx+1]]
    if len(train_items) == 0:
        return []
    #User profile = average audio embedding of items they've played
    profile = item_features_scaled[train_items].mean(axis=0, keepdims=True)
    distances, neighbors = nn_model.kneighbors(profile, n_neighbors=K + len(train_items))
    #Exclude items already seen in training
    recs = [i for i in neighbors[0] if i not in set(train_items)]
    return recs[:K]

In [ ]:
def evaluate_content_baseline_fast(train_matrix, test_matrix, item_features_scaled, nn_model, K=10):
    train_matrix = train_matrix.tocsr()
    test_matrix = test_matrix.tocsr()
    n_users = train_matrix.shape[0]

    #Build all user profiles at once
    profiles = np.zeros((n_users, item_features_scaled.shape[1]), dtype=np.float32)
    valid_users = []
    for u in range(n_users):
        items = train_matrix.indices[train_matrix.indptr[u]:train_matrix.indptr[u+1]]
        if len(items) > 0:
            profiles[u] = item_features_scaled[items].mean(axis=0)
            valid_users.append(u)

    # One batched query instead of thousands of individual ones
    distances, neighbors = nn_model.kneighbors(profiles[valid_users], n_neighbors=K + 50)

    precisions, recalls, ndcgs = [], [], []
    for i, u in enumerate(valid_users):
        test_items = set(test_matrix.indices[test_matrix.indptr[u]:test_matrix.indptr[u+1]])
        if not test_items:
            continue
        seen = set(train_matrix.indices[train_matrix.indptr[u]:train_matrix.indptr[u+1]])
        recs = [n for n in neighbors[i] if n not in seen][:K]
        if not recs:
            continue
        hits = [1 if item in test_items else 0 for item in recs]
        precisions.append(sum(hits) / K)
        recalls.append(sum(hits) / len(test_items))
        dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
        idcg = sum(1 / np.log2(j + 2) for j in range(min(len(test_items), K)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

In [ ]:
precision_c, recall_c, ndcg_c = evaluate_content_baseline_fast(
    train_matrix, test_matrix, item_features_scaled, nn_model, K=K
)

print(f"Precision@{K}: {precision_c:.4f}")
print(f"Recall@{K}: {recall_c:.4f}")
print(f"NDCG@{K}: {ndcg_c:.4f}")

Precision@10: 0.0545
Recall@10: 0.0008
NDCG@10: 0.0570


In [ ]:
results_content = {
    "model": "Pure Content (audio k-NN)",
    "K": K,
    "precision_at_k": precision_c,
    "ndcg_at_k": ndcg_c,
    "recall_at_k": recall_c,
}

#Append to your existing CF results file
existing = pd.read_csv(os.path.join(SUBSAMPLE_DIR, "baseline_pure_cf_results.csv"))
combined = pd.concat([existing, pd.DataFrame([results_content])], ignore_index=True)
combined.to_csv(os.path.join(SUBSAMPLE_DIR, "baseline_comparison.csv"), index=False)

print(combined[["model", "precision_at_k", "recall_at_k", "ndcg_at_k"]])

                       model  precision_at_k  recall_at_k  ndcg_at_k
0     Pure CF (implicit BPR)         0.53910     0.008171   0.557960
1  Pure Content (audio k-NN)         0.05445     0.000792   0.057045


Two tower Model - Using PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def bpr_loss(pos_scores, neg_scores):
    return -F.logsigmoid(pos_scores - neg_scores).mean()

def sample_training_batch(train_matrix, batch_size, n_items, device):
    coo = train_matrix.tocoo()
    idx = np.random.randint(0, len(coo.data), size=batch_size)
    users = coo.row[idx]
    pos_items = coo.col[idx]
    neg_items = np.random.randint(0, n_items, size=batch_size)
    return (torch.tensor(users, device=device),
            torch.tensor(pos_items, device=device),
            torch.tensor(neg_items, device=device))

In [ ]:
def build_history_sparse(train_m, n_users, n_items, device):
    coo = train_m.tocoo()
    counts = np.maximum(np.bincount(coo.row, minlength=n_users), 1)
    values = (1.0 / counts[coo.row]).astype(np.float32)
    idx = torch.tensor(np.vstack([coo.row, coo.col]), dtype=torch.long, device=device)
    val = torch.tensor(values, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(idx, val, size=(n_users, n_items), device=device).coalesce()


class UserTower(nn.Module):
    def __init__(self, n_users, embedding_dim=64):
        super().__init__()
        self.id_embedding = nn.Embedding(n_users, embedding_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.history_proj = nn.Linear(embedding_dim, embedding_dim)
        self.combine = nn.Linear(embedding_dim * 2, embedding_dim)
        nn.init.normal_(self.id_embedding.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        self.register_buffer("history_emb", torch.zeros(n_users, embedding_dim), persistent=False)

    def refresh_history(self, history_sparse, item_embedding_matrix):
        with torch.no_grad():
            self.history_emb = torch.sparse.mm(history_sparse, item_embedding_matrix.detach())

    def forward(self, user_idx):
        id_emb = self.id_embedding(user_idx)
        hist_emb = self.history_proj(self.history_emb[user_idx])
        combined = self.combine(torch.cat([id_emb, hist_emb], dim=-1))
        bias = self.user_bias(user_idx).squeeze(-1)
        return combined, bias

In [ ]:
class TwoTowerModel(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64):
        super().__init__()
        self.user_tower = UserTower(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx):
        u, u_bias = self.user_tower(user_idx)
        i = self.item_embedding(item_idx)
        i_bias = self.item_bias(item_idx).squeeze(-1)
        return (u * i).sum(dim=-1) + u_bias + i_bias

In [ ]:
def evaluate_two_tower(model, train_matrix, test_matrix, n_items, K=10, batch_size=200, device="cuda"):
    model.eval()
    train_matrix = train_matrix.tocsr()
    test_matrix = test_matrix.tocsr()
    n_users = train_matrix.shape[0]

    precisions, recalls, ndcgs = [], [], []
    all_items = torch.arange(n_items, device=device)

    with torch.no_grad():
        item_emb_all = model.item_embedding(all_items)
        item_bias_all = model.item_bias(all_items).squeeze(-1)

        for start in range(0, n_users, batch_size):
            end = min(start + batch_size, n_users)
            user_batch = torch.arange(start, end, device=device)
            user_emb, user_bias = model.user_tower(user_batch)

            scores = user_emb @ item_emb_all.T
            scores = scores + item_bias_all.unsqueeze(0) + user_bias.unsqueeze(1)

            for i, u in enumerate(range(start, end)):
                test_items = set(test_matrix.indices[test_matrix.indptr[u]:test_matrix.indptr[u+1]])
                if not test_items:
                    continue
                train_items = train_matrix.indices[train_matrix.indptr[u]:train_matrix.indptr[u+1]]
                user_scores = scores[i].clone()
                user_scores[train_items] = -float("inf")
                top_k = torch.topk(user_scores, K).indices.cpu().numpy()
                hits = [1 if item in test_items else 0 for item in top_k]
                precisions.append(sum(hits) / K)
                recalls.append(sum(hits) / len(test_items))
                dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
                idcg = sum(1 / np.log2(j + 2) for j in range(min(len(test_items), K)))
                ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Start with 5 for speed while testing; change to list(range(30)) once you're
# confident the pipeline runs cleanly end-to-end, and have time budget for it.
SEEDS = [0, 1, 2, 3, 4]

In [ ]:
from implicit.evaluation import train_test_split
from scipy.sparse import csr_matrix

def build_split(seed):
    set_seed(seed)
    matrix = csr_matrix(
        (subsample["count"].astype(np.float32), (subsample["user_idx"], subsample["item_idx"])),
        shape=(n_users, n_items)
    )
    train_m, test_m = train_test_split(matrix, train_percentage=0.8, random_state=seed)
    return train_m, test_m

In [ ]:
from implicit.bpr import BayesianPersonalizedRanking
from implicit.evaluation import precision_at_k, ndcg_at_k

def run_pure_cf(seed, train_m, test_m, K=10):
    set_seed(seed)
    model = BayesianPersonalizedRanking(factors=64, learning_rate=0.01,
                                         regularization=0.01, iterations=100, random_state=seed)
    model.fit(train_m)
    p = precision_at_k(model, train_m, test_m, K=K)
    n = ndcg_at_k(model, train_m, test_m, K=K)
    r = recall_at_k(model, train_m, test_m, K=K)  # your existing function
    return p, r, n

In [ ]:
def run_pure_content(seed, train_m, test_m, K=10):
    set_seed(seed)
    # nn_model/item_features_scaled don't depend on the split — only which
    # train items build each user's profile does, so results still vary by seed.
    return evaluate_content_baseline_fast(train_m, test_m, item_features_scaled, nn_model, K=K)

In [ ]:
def run_two_tower(seed, train_m, test_m, K=10, n_epochs=100, steps_per_epoch=500, batch_size=1024):
    set_seed(seed)
    model = TwoTowerModel(n_users, n_items, embedding_dim=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-6)

    for epoch in range(n_epochs):
        for _ in range(steps_per_epoch):
            users, pos_items, neg_items = sample_training_batch(train_m, batch_size, n_items, device)
            pos_scores = model(users, pos_items)
            neg_scores = model(users, neg_items)
            loss = bpr_loss(pos_scores, neg_scores)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    p, r, n = evaluate_two_tower(model, train_m, test_m, n_items, K=K, device=device)
    return p, r, n

In [ ]:
results_path = os.path.join(SUBSAMPLE_DIR, "multi_seed_results.csv")

if os.path.exists(results_path):
    multi_run_df = pd.read_csv(results_path)
    print("Loaded saved multi-seed results, skipped retraining.")
else:
  records = []

  for seed in SEEDS:
      print(f"=== Seed {seed} ===")
      train_m, test_m = build_split(seed)

      p, r, n = run_pure_cf(seed, train_m, test_m)
      records.append({"seed": seed, "model": "Pure CF (implicit BPR)", "precision_at_k": p, "recall_at_k": r, "ndcg_at_k": n})
      print(f"  Pure CF:      P={p:.4f} R={r:.4f} N={n:.4f}")

      p, r, n = run_pure_content(seed, train_m, test_m)
      records.append({"seed": seed, "model": "Pure Content (audio k-NN)", "precision_at_k": p, "recall_at_k": r, "ndcg_at_k": n})
      print(f"  Pure Content: P={p:.4f} R={r:.4f} N={n:.4f}")

      p, r, n = run_two_tower(seed, train_m, test_m)
      records.append({"seed": seed, "model": "Two-Tower CF-only", "precision_at_k": p, "recall_at_k": r, "ndcg_at_k": n})
      print(f"  Two-Tower:    P={p:.4f} R={r:.4f} N={n:.4f}")

  multi_run_df = pd.DataFrame(records)
  multi_run_df.to_csv(os.path.join(SUBSAMPLE_DIR, "multi_seed_results.csv"), index=False)

Loaded saved multi-seed results, skipped retraining.


In [ ]:
summary = multi_run_df.groupby("model")[["precision_at_k", "recall_at_k", "ndcg_at_k"]].agg(["mean", "std"])
print(summary)

                          precision_at_k           recall_at_k            \
                                    mean       std        mean       std   
model                                                                      
Pure CF (implicit BPR)           0.53359  0.001456    0.008087  0.000021   
Pure Content (audio k-NN)        0.05676  0.001622    0.000827  0.000028   
Two-Tower CF-only                0.47433  0.003013    0.007096  0.000049   

                          ndcg_at_k            
                               mean       std  
model                                          
Pure CF (implicit BPR)     0.550241  0.001670  
Pure Content (audio k-NN)  0.059443  0.001497  
Two-Tower CF-only          0.492546  0.002418  


In [ ]:
from scipy.stats import wilcoxon

def paired_test(df, model_a, model_b, metric="precision_at_k"):
    a = df[df["model"] == model_a].sort_values("seed")[metric].values
    b = df[df["model"] == model_b].sort_values("seed")[metric].values
    stat, p_value = wilcoxon(a, b)
    print(f"{model_a} vs {model_b} ({metric}): W={stat:.4f}, p={p_value:.4g}")
    return p_value

paired_test(multi_run_df, "Two-Tower CF-only", "Pure CF (implicit BPR)")
paired_test(multi_run_df, "Two-Tower CF-only", "Pure Content (audio k-NN)")

Two-Tower CF-only vs Pure CF (implicit BPR) (precision_at_k): W=0.0000, p=0.0625
Two-Tower CF-only vs Pure Content (audio k-NN) (precision_at_k): W=0.0000, p=0.0625


np.float64(0.0625)

Adding fusion to the item tower

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
mfcc_subsample = pd.read_csv(os.path.join(SUBSAMPLE_DIR, "mfcc_subsample.csv"), index_col="track_id")

feature_cols = mfcc_subsample.columns.tolist()
audio_dim = len(feature_cols)

audio_features = np.zeros((n_items, audio_dim), dtype=np.float32)
missing = 0
for idx, track_id in item_id_map.items():
    if track_id in mfcc_subsample.index:
        audio_features[idx] = mfcc_subsample.loc[track_id, feature_cols].values
    else:
        missing += 1

print(f"Audio feature matrix: {audio_features.shape}, {missing} items missing (should be 0)")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
audio_features_scaled = scaler.fit_transform(audio_features).astype(np.float32)
audio_features_tensor = torch.tensor(audio_features_scaled, device=device)
print("Audio tensor ready:", audio_features_tensor.shape)

Audio feature matrix: (56377, 104), 0 items missing (should be 0)
Audio tensor ready: torch.Size([56377, 104])


In [ ]:
class HybridItemTower(nn.Module):
    def __init__(self, n_items, audio_dim, embedding_dim=64, fusion="concat"):
        super().__init__()
        self.fusion = fusion
        self.cf_embedding = nn.Embedding(n_items, embedding_dim)
        self.item_bias = nn.Embedding(n_items, 1)
        self.audio_proj = nn.Linear(audio_dim, embedding_dim)

        if fusion == "concat":
            self.fusion_proj = nn.Linear(embedding_dim * 2, embedding_dim)
        elif fusion == "weighted_sum":
            self.alpha = nn.Parameter(torch.tensor(0.5))
        elif fusion == "gated":
            self.gate = nn.Linear(embedding_dim * 2, embedding_dim)
        else:
            raise ValueError(f"Unknown fusion type: {fusion}")

        nn.init.normal_(self.cf_embedding.weight, std=0.01)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, item_idx, audio_features_tensor):
        cf_emb = self.cf_embedding(item_idx)
        audio_emb = self.audio_proj(audio_features_tensor[item_idx])

        if self.fusion == "concat":
            fused = self.fusion_proj(torch.cat([cf_emb, audio_emb], dim=-1))
        elif self.fusion == "weighted_sum":
            a = torch.sigmoid(self.alpha)
            fused = a * cf_emb + (1 - a) * audio_emb
        elif self.fusion == "gated":
            gate_weight = torch.sigmoid(self.gate(torch.cat([cf_emb, audio_emb], dim=-1)))
            fused = gate_weight * cf_emb + (1 - gate_weight) * audio_emb

        bias = self.item_bias(item_idx).squeeze(-1)
        return fused, bias


class HybridTwoTowerModel(nn.Module):
    def __init__(self, n_users, n_items, audio_dim, embedding_dim=64, fusion="concat"):
        super().__init__()
        self.user_tower = UserTower(n_users, embedding_dim)
        self.item_tower = HybridItemTower(n_items, audio_dim, embedding_dim, fusion=fusion)

    def forward(self, user_idx, item_idx, audio_features_tensor):
        u, u_bias = self.user_tower(user_idx)
        item_emb, item_bias = self.item_tower(item_idx, audio_features_tensor)
        return (u * item_emb).sum(dim=-1) + u_bias + item_bias

In [ ]:
def build_cold_start_split(seed, cold_user_frac=0.10, cold_item_frac=0.10, cold_train_range=(3, 5)):
    rng = np.random.RandomState(seed)
    full = csr_matrix((subsample["count"].astype(np.float32),
                       (subsample["user_idx"], subsample["item_idx"])),
                      shape=(n_users, n_items)).tocsr()
    full.sum_duplicates(); full.sort_indices()
    coo = full.tocoo()
    rows, cols, vals = coo.row, coo.col, coo.data   # same order as full.indices/indptr

    cold_users = rng.choice(n_users, int(cold_user_frac * n_users), replace=False)
    cold_items = rng.choice(n_items, int(cold_item_frac * n_items), replace=False)
    is_cold_user = np.zeros(n_users, dtype=bool); is_cold_user[cold_users] = True
    is_cold_item = np.zeros(n_items, dtype=bool); is_cold_item[cold_items] = True

    warm_item_inter = ~is_cold_item[cols]
    cold_user_inter = is_cold_user[rows]

    # Warm users: 80% of warm-item interactions go to train
    train_mask = (~cold_user_inter) & warm_item_inter & (rng.rand(len(rows)) < 0.8)

    # Cold users: keep only 3-5 warm-item interactions in train
    for u in cold_users:
        pos = np.arange(full.indptr[u], full.indptr[u + 1])
        pos = pos[warm_item_inter[pos]]
        k = min(rng.randint(cold_train_range[0], cold_train_range[1] + 1), max(len(pos) - 1, 0))
        if k > 0:
            train_mask[rng.choice(pos, size=k, replace=False)] = True

    def sub(mask):
        return csr_matrix((vals[mask], (rows[mask], cols[mask])), shape=(n_users, n_items))

    split = {
        "train": sub(train_mask),
        "test_warm": sub(~train_mask & ~cold_user_inter & warm_item_inter),
        "test_ucold": sub(~train_mask & cold_user_inter & warm_item_inter),
        "test_icold": sub(~cold_user_inter & ~warm_item_inter),
        "warm_item_mask": ~is_cold_item,
        "cold_item_mask": is_cold_item,
    }
    print(f"seed {seed}: train={split['train'].nnz:,} | test warm={split['test_warm'].nnz:,} | "
          f"user-cold={split['test_ucold'].nnz:,} | item-cold={split['test_icold'].nnz:,}")
    return split

In [ ]:
def evaluate_split(score_fn, split, K=10, batch_size=200, max_test=50):
    train_m = split["train"].tocsr()
    groups = {
        "warm":      (split["test_warm"],  split["warm_item_mask"]),
        "user_cold": (split["test_ucold"], split["warm_item_mask"]),
        "item_cold": (split["test_icold"], split["cold_item_mask"]),
    }
    out = {}
    for gname, (test_m, cand_mask) in groups.items():
        test_m = test_m.tocsr()
        users = np.where(np.diff(test_m.indptr) > 0)[0]
        cand = torch.as_tensor(cand_mask, device=device)
        rng = np.random.RandomState(12345)   # identical capped test items for every model
        P, R, N = [], [], []
        for s in range(0, len(users), batch_size):
            ub = users[s:s + batch_size]
            scores = score_fn(ub).masked_fill(~cand.unsqueeze(0), -float("inf"))
            for i, u in enumerate(ub):
                items = test_m.indices[test_m.indptr[u]:test_m.indptr[u + 1]]
                if len(items) > max_test:
                    items = rng.choice(items, max_test, replace=False)
                test_items = set(items)
                seen = torch.from_numpy(train_m.indices[train_m.indptr[u]:train_m.indptr[u + 1]]).long().to(device)
                sc = scores[i].clone(); sc[seen] = -float("inf")
                top = torch.topk(sc, K).indices.cpu().numpy()
                hits = [1 if it in test_items else 0 for it in top]
                P.append(sum(hits) / K)
                R.append(sum(hits) / len(test_items))
                dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
                idcg = sum(1 / np.log2(j + 2) for j in range(min(len(test_items), K)))
                N.append(dcg / idcg if idcg > 0 else 0.0)
        out[gname] = (np.mean(P), np.mean(R), np.mean(N), len(users))
    return out

In [ ]:
def scorer_implicit(model):
    def to_np(x): return x.to_numpy() if hasattr(x, "to_numpy") else np.asarray(x)
    UF = torch.tensor(to_np(model.user_factors), device=device, dtype=torch.float32)
    IF = torch.tensor(to_np(model.item_factors), device=device, dtype=torch.float32)  # includes bias column
    return lambda ub: UF[torch.as_tensor(ub, device=device).long()] @ IF.T

def scorer_content(train_m):
    train_m = train_m.tocsr()
    feats = audio_features_tensor
    feats_n = F.normalize(feats, dim=1)
    def fn(ub):
        profs = []
        for u in ub:
            items = train_m.indices[train_m.indptr[u]:train_m.indptr[u + 1]]
            if len(items) == 0:
                profs.append(torch.zeros(feats.shape[1], device=device))
            else:
                profs.append(feats[torch.from_numpy(items).long().to(device)].mean(0))
        return F.normalize(torch.stack(profs), dim=1) @ feats_n.T
    return fn

def scorer_two_tower(model):
    model.eval()
    with torch.no_grad():
        ie = model.item_embedding.weight; ib = model.item_bias.weight.squeeze(-1)
    def fn(ub):
        with torch.no_grad():
            u_idx = torch.as_tensor(ub, device=device).long()
            u, u_bias = model.user_tower(u_idx)
            return u @ ie.T + ib.unsqueeze(0) + u_bias.unsqueeze(1)
    return fn

def scorer_hybrid(model):
    model.eval()
    with torch.no_grad():
        ie, ib = model.item_tower(torch.arange(n_items, device=device), audio_features_tensor)
    def fn(ub):
        with torch.no_grad():
            u_idx = torch.as_tensor(ub, device=device).long()
            u, u_bias = model.user_tower(u_idx)
            return u @ ie.T + ib.unsqueeze(0) + u_bias.unsqueeze(1)
    return fn

In [ ]:
class CFDropoutEmbedding(nn.Module):
    """Randomly zeroes the CF embedding during training so the item tower learns to use audio alone.
    Same idea as DropoutNet [6]. Only used if cf_dropout > 0."""
    def __init__(self, emb, p):
        super().__init__(); self.emb = emb; self.p = p
    def forward(self, idx):
        out = self.emb(idx)
        if self.training and self.p > 0:
            out = out * (torch.rand(out.shape[0], 1, device=out.device) > self.p).float()
        return out

def make_hybrid(fusion, cf_dropout=0.0):
    m = HybridTwoTowerModel(n_users, n_items, audio_dim, 64, fusion).to(device)
    if cf_dropout > 0:
        m.item_tower.cf_embedding = CFDropoutEmbedding(m.item_tower.cf_embedding, cf_dropout)
    return m

def train_neural(model, fwd, split, get_item_emb_fn=None, n_epochs=60, steps_per_epoch=500, batch_size=1024, lr=0.005):
    coo = split["train"].tocoo()
    warm_items = np.where(split["warm_item_mask"])[0]
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)

    history_sparse = None
    if get_item_emb_fn is not None:
        history_sparse = build_history_sparse(split["train"], n_users, n_items, device)

    model.train()
    for epoch in range(n_epochs):
        if get_item_emb_fn is not None:
            with torch.no_grad():
                model.user_tower.refresh_history(history_sparse, get_item_emb_fn())
        total = 0
        for _ in range(steps_per_epoch):
            idx = np.random.randint(0, len(coo.data), size=batch_size)
            users = torch.as_tensor(coo.row[idx], device=device).long()
            pos = torch.as_tensor(coo.col[idx], device=device).long()
            neg = torch.as_tensor(warm_items[np.random.randint(0, len(warm_items), size=batch_size)], device=device).long()
            loss = bpr_loss(fwd(users, pos), fwd(users, neg))
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        if (epoch + 1) % 20 == 0:
            print(f"    epoch {epoch+1}: loss={total/steps_per_epoch:.4f}")

    if get_item_emb_fn is not None:
        with torch.no_grad():
            model.user_tower.refresh_history(history_sparse, get_item_emb_fn())
    return model

In [ ]:
import time
CS_RESULTS = os.path.join(SUBSAMPLE_DIR, "cold_start_results.csv")

def run_cold_start_sweep(seeds, K=10):
    done = pd.read_csv(CS_RESULTS) if os.path.exists(CS_RESULTS) else pd.DataFrame(
        columns=["seed", "model", "group", "precision_at_k", "recall_at_k", "ndcg_at_k", "n_users"])
    for seed in seeds:
        split = build_cold_start_split(seed)
        jobs = ["Pure CF (implicit BPR)", "Pure Content (audio k-NN)", "Two-Tower CF-only",
                "Hybrid concat", "Hybrid weighted_sum", "Hybrid gated"]
        for name in jobs:
            label = name + (f" +CFdrop{CF_DROPOUT}" if CF_DROPOUT > 0 and name.startswith("Hybrid") else "")
            if ((done["seed"] == seed) & (done["model"] == label)).any():
                print(f"  skip {label} seed {seed} (already saved)")
                continue
            t0 = time.time()
            set_seed(seed)

            if name.startswith("Pure CF"):
                m = BayesianPersonalizedRanking(factors=64, learning_rate=0.01, regularization=0.01,
                                                 iterations=100, random_state=seed)
                m.fit(split["train"])
                fn = scorer_implicit(m)
            elif name.startswith("Pure Content"):
                fn = scorer_content(split["train"])
            elif name == "Two-Tower CF-only":
                m = TwoTowerModel(n_users, n_items, embedding_dim=64).to(device)
                train_neural(m, lambda u, i: m(u, i), split,
                              get_item_emb_fn=lambda: m.item_embedding.weight)
                fn = scorer_two_tower(m)
            else:
                fusion = name.split()[1]
                m = make_hybrid(fusion, CF_DROPOUT)
                train_neural(m, lambda u, i: m(u, i, audio_features_tensor), split,
                              get_item_emb_fn=lambda: m.item_tower(torch.arange(n_items, device=device), audio_features_tensor)[0])
                fn = scorer_hybrid(m)

            res = evaluate_split(fn, split, K=K)
            rows = [{"seed": seed, "model": label, "group": g, "precision_at_k": v[0],
                     "recall_at_k": v[1], "ndcg_at_k": v[2], "n_users": v[3]} for g, v in res.items()]
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
            done.to_csv(CS_RESULTS, index=False)
            print(f"  {label} seed {seed}: " + " | ".join(f"{g} P={v[0]:.4f}" for g, v in res.items())
                  + f"  ({time.time()-t0:.0f}s)")

In [ ]:
SEEDS_CS = list(range(30))
CF_DROPOUT = 0.0; run_cold_start_sweep(SEEDS_CS)
CF_DROPOUT = 0.3; run_cold_start_sweep(SEEDS_CS)

seed 0: train=4,467,632 | test warm=1,116,417 | user-cold=616,819 | item-cold=603,487
  skip Pure CF (implicit BPR) seed 0 (already saved)
  skip Pure Content (audio k-NN) seed 0 (already saved)
  skip Two-Tower CF-only seed 0 (already saved)
  skip Hybrid concat seed 0 (already saved)
  skip Hybrid weighted_sum seed 0 (already saved)
  skip Hybrid gated seed 0 (already saved)
seed 1: train=4,464,929 | test warm=1,116,622 | user-cold=607,292 | item-cold=615,486
  skip Pure CF (implicit BPR) seed 1 (already saved)
  skip Pure Content (audio k-NN) seed 1 (already saved)
  skip Two-Tower CF-only seed 1 (already saved)
  skip Hybrid concat seed 1 (already saved)
  skip Hybrid weighted_sum seed 1 (already saved)
  skip Hybrid gated seed 1 (already saved)
seed 2: train=4,451,845 | test warm=1,112,695 | user-cold=611,811 | item-cold=626,161
  skip Pure CF (implicit BPR) seed 2 (already saved)
  skip Pure Content (audio k-NN) seed 2 (already saved)
  skip Two-Tower CF-only seed 2 (already save

  0%|          | 0/100 [00:00<?, ?it/s]

  Pure CF (implicit BPR) seed 10: warm P=0.0452 | user_cold P=0.0115 | item_cold P=0.0096  (34s)
  Pure Content (audio k-NN) seed 10: warm P=0.0043 | user_cold P=0.0005 | item_cold P=0.0279  (2s)


/tmp/ipykernel_28902/2171941275.py:7: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, val, size=(n_users, n_items), device=device).coalesce()


    epoch 20: loss=0.2978
    epoch 40: loss=0.2774
    epoch 60: loss=0.2734
  Two-Tower CF-only seed 10: warm P=0.0350 | user_cold P=0.0060 | item_cold P=0.0101  (109s)
    epoch 20: loss=0.3110
    epoch 40: loss=0.2968
    epoch 60: loss=0.2946
  Hybrid concat seed 10: warm P=0.0316 | user_cold P=0.0060 | item_cold P=0.0149  (131s)
    epoch 20: loss=0.2921
    epoch 40: loss=0.2751
    epoch 60: loss=0.2714
  Hybrid weighted_sum seed 10: warm P=0.0374 | user_cold P=0.0075 | item_cold P=0.0174  (131s)
    epoch 20: loss=0.2934
    epoch 40: loss=0.2855
    epoch 60: loss=0.2754
  Hybrid gated seed 10: warm P=0.0327 | user_cold P=0.0050 | item_cold P=0.0212  (142s)
seed 11: train=4,464,914 | test warm=1,116,618 | user-cold=607,716 | item-cold=615,218


...

  Pure CF (implicit BPR) seed 27: warm P=0.0434 | user_cold P=0.0110 | item_cold P=0.0088  (30s)
  Pure Content (audio k-NN) seed 27: warm P=0.0053 | user_cold P=0.0015 | item_cold P=0.0239  (2s)
    epoch 20: loss=0.2960
    epoch 40: loss=0.2798
    epoch 60: loss=0.2725
  Two-Tower CF-only seed 27: warm P=0.0383 | user_cold P=0.0050 | item_cold P=0.0092  (102s)
    epoch 20: loss=0.3100
    epoch 40: loss=0.2981
    epoch 60: loss=0.2939
  Hybrid concat seed 27: warm P=0.0310 | user_cold P=0.0055 | item_cold P=0.0135  (129s)
    epoch 20: loss=0.2921
    epoch 40: loss=0.2778
    epoch 60: loss=0.2710
  Hybrid weighted_sum seed 27: warm P=0.0355 | user_cold P=0.0045 | item_cold P=0.0169  (131s)
    epoch 20: loss=0.2964
    epoch 40: loss=0.2796
    epoch 60: loss=0.2731
  Hybrid gated seed 27: warm P=0.0323 | user_cold P=0.0080 | item_cold P=0.0263  (142s)
seed 28: train=4,452,734 | test warm=1,113,234 | user-cold=630,683 | item-cold=605,810


  0%|          | 0/100 [00:00<?, ?it/s]

  Pure CF (implicit BPR) seed 28: warm P=0.0445 | user_cold P=0.0190 | item_cold P=0.0041  (30s)
  Pure Content (audio k-NN) seed 28: warm P=0.0043 | user_cold P=0.0035 | item_cold P=0.0271  (2s)
    epoch 20: loss=0.2973
    epoch 40: loss=0.2793
    epoch 60: loss=0.2741
  Two-Tower CF-only seed 28: warm P=0.0352 | user_cold P=0.0055 | item_cold P=0.0091  (103s)
    epoch 20: loss=0.3122
    epoch 40: loss=0.2989
    epoch 60: loss=0.2951
  Hybrid concat seed 28: warm P=0.0288 | user_cold P=0.0065 | item_cold P=0.0141  (128s)
    epoch 20: loss=0.2937
    epoch 40: loss=0.2777
    epoch 60: loss=0.2726
  Hybrid weighted_sum seed 28: warm P=0.0359 | user_cold P=0.0075 | item_cold P=0.0149  (131s)
    epoch 20: loss=0.2950
    epoch 40: loss=0.2802
    epoch 60: loss=0.2776
  Hybrid gated seed 28: warm P=0.0284 | user_cold P=0.0055 | item_cold P=0.0192  (141s)
seed 29: train=4,458,190 | test warm=1,114,733 | user-cold=614,357 | item-cold=615,916


  0%|          | 0/100 [00:00<?, ?it/s]

  Pure CF (implicit BPR) seed 29: warm P=0.0451 | user_cold P=0.0100 | item_cold P=0.0053  (31s)
  Pure Content (audio k-NN) seed 29: warm P=0.0043 | user_cold P=0.0010 | item_cold P=0.0243  (2s)
    epoch 20: loss=0.2997
    epoch 40: loss=0.2799
    epoch 60: loss=0.2754
  Two-Tower CF-only seed 29: warm P=0.0380 | user_cold P=0.0080 | item_cold P=0.0078  (102s)
    epoch 20: loss=0.3148
    epoch 40: loss=0.2992
    epoch 60: loss=0.2973
  Hybrid concat seed 29: warm P=0.0293 | user_cold P=0.0060 | item_cold P=0.0106  (130s)
    epoch 20: loss=0.2943
    epoch 40: loss=0.2778
    epoch 60: loss=0.2736
  Hybrid weighted_sum seed 29: warm P=0.0382 | user_cold P=0.0080 | item_cold P=0.0122  (132s)
    epoch 20: loss=0.2985
    epoch 40: loss=0.2806
    epoch 60: loss=0.2782
  Hybrid gated seed 29: warm P=0.0328 | user_cold P=0.0065 | item_cold P=0.0223  (143s)
seed 0: train=4,467,632 | test warm=1,116,417 | user-cold=616,819 | item-cold=603,487
  skip Pure CF (implicit BPR) seed 0 (alr

In [ ]:
cs = pd.read_csv(CS_RESULTS)

def report_table(metric):
    g = cs.groupby(["model", "group"])[metric].agg(["mean", "std"])
    t = (g["mean"].map("{:.4f}".format) + " ± " + g["std"].map("{:.4f}".format)).unstack("group")
    return t[["warm", "user_cold", "item_cold"]]

for m in ["precision_at_k", "recall_at_k", "ndcg_at_k"]:
    t = report_table(m); print(m); print(t, "\n")
    t.to_csv(os.path.join(SUBSAMPLE_DIR, f"report_table_{m}.csv"))

means = cs.groupby(["group", "model"])["ndcg_at_k"].mean().unstack("group")
print((means / means.max() * 100).round(0))   # 100 = best model in that group

for f in ["concat", "weighted_sum", "gated"]:
    a = cs[(cs.model == f"Hybrid {f}") & (cs.group == "item_cold")].sort_values("seed")["precision_at_k"].values
    b = cs[(cs.model == f"Hybrid {f} +CFdrop0.3") & (cs.group == "item_cold")].sort_values("seed")["precision_at_k"].values
    print(f"{f}: CF-dropout better in {(b > a).sum()}/{len(a)} seeds, p={wilcoxon(a, b).pvalue:.4f}")

precision_at_k
group                                      warm        user_cold  \
model                                                              
Hybrid concat                   0.0302 ± 0.0021  0.0065 ± 0.0022   
Hybrid concat +CFdrop0.3        0.0286 ± 0.0020  0.0060 ± 0.0016   
Hybrid gated                    0.0328 ± 0.0036  0.0061 ± 0.0021   
Hybrid gated +CFdrop0.3         0.0312 ± 0.0024  0.0062 ± 0.0015   
Hybrid weighted_sum             0.0368 ± 0.0014  0.0063 ± 0.0014   
Hybrid weighted_sum +CFdrop0.3  0.0364 ± 0.0016  0.0056 ± 0.0018   
Pure CF (implicit BPR)          0.0442 ± 0.0018  0.0130 ± 0.0029   
Pure Content (audio k-NN)       0.0045 ± 0.0005  0.0019 ± 0.0008   
Two-Tower CF-only               0.0371 ± 0.0014  0.0068 ± 0.0017   

group                                 item_cold  
model                                            
Hybrid concat                   0.0135 ± 0.0015  
Hybrid concat +CFdrop0.3        0.0146 ± 0.0019  
Hybrid gated                    0.02

In [ ]:
from statsmodels.stats.multitest import multipletests
cs = pd.read_csv(CS_RESULTS)

def cmp(a_name, b_name, group, metric="ndcg_at_k"):
    a = cs[(cs.model == a_name) & (cs.group == group)].sort_values("seed")[metric].values
    b = cs[(cs.model == b_name) & (cs.group == group)].sort_values("seed")[metric].values
    return {"group": group, "comparison": f"{a_name} vs {b_name}",
            "wins": f"{(a > b).sum()}/{len(a)}", "p": wilcoxon(a, b).pvalue}

rows = [cmp(h, "Two-Tower CF-only", g) for g in ["warm", "user_cold", "item_cold"]
        for h in ["Hybrid concat", "Hybrid weighted_sum", "Hybrid gated"]]
rows += [cmp(h, "Pure Content (audio k-NN)", "item_cold") for h in ["Hybrid gated", "Hybrid gated +CFdrop0.3"]]

res = pd.DataFrame(rows)
res["p_holm"] = multipletests(res["p"], method="holm")[1]
print(res.round(4).to_string(index=False))
res.to_csv(os.path.join(SUBSAMPLE_DIR, "significance_tests.csv"), index=False)

    group                                           comparison  wins      p  p_holm
     warm                   Hybrid concat vs Two-Tower CF-only  0/30 0.0000  0.0000
     warm             Hybrid weighted_sum vs Two-Tower CF-only 13/30 0.3085  0.3085
     warm                    Hybrid gated vs Two-Tower CF-only  1/30 0.0000  0.0000
user_cold                   Hybrid concat vs Two-Tower CF-only  9/30 0.0497  0.1988
user_cold             Hybrid weighted_sum vs Two-Tower CF-only  9/30 0.0364  0.1822
user_cold                    Hybrid gated vs Two-Tower CF-only 11/30 0.0879  0.2638
item_cold                   Hybrid concat vs Two-Tower CF-only 30/30 0.0000  0.0000
item_cold             Hybrid weighted_sum vs Two-Tower CF-only 30/30 0.0000  0.0000
item_cold                    Hybrid gated vs Two-Tower CF-only 30/30 0.0000  0.0000
item_cold            Hybrid gated vs Pure Content (audio k-NN)  3/30 0.0000  0.0000
item_cold Hybrid gated +CFdrop0.3 vs Pure Content (audio k-NN) 13/30 0.1094 